# Notebook 2: Indexing & Retrieval
## Chronicling America 1906 Newspaper Corpus

Builds a PyTerrier inverted index over the preprocessed corpus, inspects term statistics, and runs BM25 and TF-IDF retrieval.

## Setup

In [8]:
# !pip install python-terrier pandas pyarrow
import os
import pandas as pd
from pathlib import Path

os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-25.jdk/Contents/Home"
os.environ["JVM_PATH"]  = "/Library/Java/JavaVirtualMachines/temurin-25.jdk/Contents/Home/lib/server/libjvm.dylib"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

import pyterrier as pt
if not pt.java.started():
    pt.java.init()

print("pyterrier ready")

pyterrier ready


## Config

In [9]:
CORPUS     = Path("../data/corpus.parquet")
INDEX_PATH = "../data/pt_index"

os.makedirs(INDEX_PATH, exist_ok=True)

## Load Corpus

In [10]:
df = pd.read_parquet(CORPUS)
print(f"loaded {len(df)} documents")
df[["doc_id", "date", "token_count"]].head(5)

loaded 608 documents


,doc_id,date,token_count
0,sn83035387_1906_01_06_ed1_seq1,1906-01-06,3239
1,sn83035387_1906_01_06_ed1_seq2,1906-01-06,3700
2,sn83035387_1906_01_06_ed1_seq3,1906-01-06,2568
3,sn83035387_1906_01_06_ed1_seq4,1906-01-06,2696
4,sn83035387_1906_01_13_ed1_seq1,1906-01-13,3260


## Build Inverted Index

In [11]:
# pyterrier expects 'docno' and 'text' columns
corpus_pt = df[["doc_id", "clean_text"]].rename(columns={"doc_id": "docno", "clean_text": "text"}).copy()
corpus_pt["docno"] = corpus_pt["docno"].astype(str)

indexer = pt.IterDictIndexer(INDEX_PATH, overwrite=True, meta={"docno": 50, "text": 25000})
index_ref = indexer.index(corpus_pt.to_dict(orient="records"))

print("index created:", index_ref)

index created: <org.terrier.querying.IndexRef at 0x17d6dbed0 jclass=org/terrier/querying/IndexRef jself=<LocalRef obj=0x16a75034a at 0x16a8324f0>>


In [12]:
index = pt.IndexFactory.of(index_ref)
stats = index.getCollectionStatistics()

print(f"documents    : {stats.getNumberOfDocuments()}")
print(f"unique terms : {stats.getNumberOfUniqueTerms():,}")
print(f"avg doc len  : {stats.getAverageDocumentLength():.2f}")

documents    : 608
unique terms : 54,172
avg doc len  : 2029.62


In [13]:
# show which documents contain a term -- demonstrates the inverted index directly
def show_postings(term, top_n=5):
    entry = index.getLexicon().getLexiconEntry(term)
    if entry is None:
        print(f"'{term}' not found")
        return
    postings = index.getInvertedIndex().getPostings(entry)
    print(f"'{term}' appears in {entry.getDocumentFrequency()} documents, showing first {top_n}:")
    for i, posting in enumerate(postings):
        if i >= top_n:
            break
        doc_id = df.iloc[posting.getId()]["doc_id"]
        print(f"  {doc_id}  (tf={posting.getFrequency()})")

show_postings("earthquak")

'earthquak' appears in 39 documents, showing first 5:
  sn83035387_1906_01_06_ed1_seq2  (tf=1)
  sn83035387_1906_03_17_ed1_seq2  (tf=1)
  sn83035387_1906_03_24_ed1_seq4  (tf=2)
  sn83035387_1906_04_21_ed1_seq4  (tf=8)
  sn83035387_1906_04_28_ed1_seq2  (tf=5)


## Posting List Inspection

Look up DF (how many documents contain the term) and TF (how many times it appears total). High DF = common term = low discriminative power for ranking.

In [14]:
def term_info(term):
    entry = index.getLexicon().getLexiconEntry(term)
    if entry is not None:
        print(f"'{term}' -> DF={entry.getDocumentFrequency()}, TF={entry.getFrequency()}")
    else:
        print(f"'{term}' not found")

# mix of common, rare, and historically relevant terms
for t in ["earthquak", "fire", "presid", "railroad", "francisco", "said"]:
    term_info(t)

'earthquak' -> DF=39, TF=76
'fire' -> DF=232, TF=675
'presid' -> DF=490, TF=2370
'railroad' -> DF=252, TF=695
'francisco' -> DF=117, TF=262
'said' not found


In [15]:
# semantically related terms -- compare coverage
for t in ["earthquak", "disaster", "flood", "storm"]:
    term_info(t)

'earthquak' -> DF=39, TF=76
'disaster' not found
'flood' -> DF=85, TF=99
'storm' -> DF=79, TF=118


## BM25 and TF-IDF Retrieval

In [16]:
bm25_pt  = pt.terrier.Retriever(index, wmodel="BM25")
tfidf_pt = pt.terrier.Retriever(index, wmodel="TF_IDF")

queries = pd.DataFrame([
    {"qid": "1", "query": "earthquake fire san francisco"},
    {"qid": "2", "query": "railroad strike labor"},
    {"qid": "3", "query": "election president congress"},
    {"qid": "4", "query": "war military troops"},
    {"qid": "5", "query": "weather storm flood"},
])

bm25_results  = bm25_pt.transform(queries)
tfidf_results = tfidf_pt.transform(queries)

print("BM25 top results:")
print(bm25_results.head(10))
print("TF-IDF top results:")
print(tfidf_results.head(10))

BM25 top results:
  qid  docid                           docno  rank      score  \
0   1     63  sn83035387_1906_04_21_ed1_seq4     0  14.774044   
1   1    347  sn84020235_1906_05_05_ed1_seq2     1  14.522445   
2   1     65  sn83035387_1906_04_28_ed1_seq2     2  14.116376   
3   1    351  sn84020235_1906_05_05_ed1_seq6     3  13.527361   
4   1    373  sn84020235_1906_05_26_ed1_seq4     4  13.508947   
5   1    562  sn84020235_1906_11_17_ed1_seq3     5  12.609765   
6   1    339  sn84020235_1906_04_28_ed1_seq2     6  12.158374   
7   1    336  sn84020235_1906_04_21_ed1_seq7     7  11.705420   
8   1    525  sn84020235_1906_10_13_ed1_seq6     8  11.654268   
9   1    133  sn83035387_1906_08_25_ed1_seq2     9  10.982038   

                           query  
0  earthquake fire san francisco  
1  earthquake fire san francisco  
2  earthquake fire san francisco  
3  earthquake fire san francisco  
4  earthquake fire san francisco  
5  earthquake fire san francisco  
6  earthquake fire sa